# BERT Fine-tuning, Training & Evaluation

This notebook covers:
1. Data loading and splitting (train/val/test)
2. BERT model fine-tuning with TensorFlow
3. Training with callbacks (early stopping, checkpoints)
4. Evaluation on test set
5. Visualization of results

## 1. Setup & Imports

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from transformers import TFBertModel, BertTokenizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath(os.path.join('..', '..')))

from src.data.bert_preprocessor import BertDataPreparer

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

## 2. Load Data & Create Train/Val/Test Split

In [ ]:
# Load processed data
df = pd.read_csv('../../data/processed/processed.csv')
print(f"Total samples: {len(df)}")
print(f"\nClass distribution:\n{df['target'].value_counts()}")
print(f"\nData info:")
df.head()

In [ ]:
# Extract texts and labels
texts = df['cleaned_text'].astype(str).values
labels = df['target'].values

# First split: 80% train+val, 20% test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

# Second split: 75% train (60% of total), 25% val (20% of total)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)

print(f"Train set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")
print(f"Test set size: {len(X_test)}")
print(f"\nTrain class distribution: {np.bincount(y_train)}")
print(f"Val class distribution: {np.bincount(y_val)}")
print(f"Test class distribution: {np.bincount(y_test)}")

## 3. Prepare Datasets using BertDataPreparer

In [ ]:
# Initialize data preparer
preparer = BertDataPreparer(model_name='bert-base-uncased')

# Create datasets
BATCH_SIZE = 32

train_dataset = preparer.create_dataset(X_train, y_train, batch_size=BATCH_SIZE, shuffle=True)
val_dataset = preparer.create_dataset(X_val, y_val, batch_size=BATCH_SIZE, shuffle=False)
test_dataset = preparer.create_dataset(X_test, y_test, batch_size=BATCH_SIZE, shuffle=False)

print("Datasets created successfully")

# Inspect batch
for inputs, labels in train_dataset.take(1):
    print(f"\nBatch inputs keys: {inputs.keys()}")
    print(f"input_word_ids shape: {inputs['input_word_ids'].shape}")
    print(f"input_mask shape: {inputs['input_mask'].shape}")
    print(f"input_type_ids shape: {inputs['input_type_ids'].shape}")
    print(f"Labels shape: {labels.shape}")

## 4. Build BERT Fine-tuning Model

In [ ]:
def build_bert_model(max_length=128, num_classes=2):
    """
    Build a BERT classification model.
    
    Args:
        max_length: Maximum sequence length
        num_classes: Number of output classes
    """
    # Load pretrained BERT model
    bert_model = TFBertModel.from_pretrained('bert-base-uncased')
    
    # Input layers
    input_word_ids = layers.Input(
        shape=(max_length,), dtype=tf.int32, name='input_word_ids'
    )
    input_mask = layers.Input(
        shape=(max_length,), dtype=tf.int32, name='input_mask'
    )
    input_type_ids = layers.Input(
        shape=(max_length,), dtype=tf.int32, name='input_type_ids'
    )
    
    # BERT processing
    bert_output = bert_model(
        input_ids=input_word_ids,
        attention_mask=input_mask,
        token_type_ids=input_type_ids,
        training=True
    )
    
    # Use [CLS] token (first token) for classification
    cls_output = bert_output.last_hidden_state[:, 0, :]
    
    # Classification head
    x = layers.Dropout(0.3)(cls_output)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    logits = layers.Dense(num_classes, activation='softmax', name='output')(x)
    
    # Create model
    model = keras.Model(
        inputs=[input_word_ids, input_mask, input_type_ids],
        outputs=logits
    )
    
    return model

# Build model
model = build_bert_model(max_length=128, num_classes=2)
print("Model built successfully")
model.summary()

## 5. Compile Model

In [ ]:
# Use a lower learning rate for fine-tuning
optimizer = keras.optimizers.Adam(learning_rate=2e-5)

model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Model compiled")

## 6. Define Callbacks

In [ ]:
# Create results directory if it doesn't exist
model_dir = '../../results/models'
os.makedirs(model_dir, exist_ok=True)

# Callbacks
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

checkpoint = keras.callbacks.ModelCheckpoint(
    filepath=os.path.join(model_dir, 'bert_best_model.h5'),
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-7,
    verbose=1
)

callbacks = [early_stopping, checkpoint, reduce_lr]
print("Callbacks defined")

## 7. Train Model

In [ ]:
# Train model
EPOCHS = 10

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print("\nTraining completed!")

## 8. Save Training History

In [ ]:
import json

# Save history
history_dict = {
    'accuracy': history.history['accuracy'],
    'loss': history.history['loss'],
    'val_accuracy': history.history['val_accuracy'],
    'val_loss': history.history['val_loss']
}

with open(os.path.join(model_dir, 'bert_training_history.json'), 'w') as f:
    json.dump(history_dict, f, indent=4)

print("Training history saved")

## 9. Evaluate on Test Set

In [ ]:
# Get predictions on test set
y_pred_probs = model.predict(test_dataset)
y_pred = np.argmax(y_pred_probs, axis=1)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("=" * 50)
print("TEST SET EVALUATION")
print("=" * 50)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print("\n" + "=" * 50)
print("CLASSIFICATION REPORT")
print("=" * 50)
print(classification_report(y_test, y_pred, target_names=['Not Fake', 'Fake']))

## 10. Confusion Matrix

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Not Fake', 'Fake'],
            yticklabels=['Not Fake', 'Fake'])
plt.title('Confusion Matrix - Test Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(os.path.join(model_dir, 'bert_confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"\nConfusion Matrix:\n{cm}")

## 11. Training Curves

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Train Accuracy', marker='o')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history.history['loss'], label='Train Loss', marker='o')
axes[1].plot(history.history['val_loss'], label='Val Loss', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Model Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(model_dir, 'bert_training_curves.png'), dpi=300, bbox_inches='tight')
plt.show()

print("Training curves saved")

## 12. Save Evaluation Results

In [ ]:
# Save evaluation results
results = {
    'test_accuracy': float(accuracy),
    'test_precision': float(precision),
    'test_recall': float(recall),
    'test_f1_score': float(f1),
    'confusion_matrix': cm.tolist(),
    'model_name': 'BERT Fine-tuned',
    'epochs_trained': len(history.history['loss'])
}

with open(os.path.join(model_dir, 'bert_evaluation_results.json'), 'w') as f:
    json.dump(results, f, indent=4)

print("\nEvaluation results saved to results/models/bert_evaluation_results.json")

## 13. Save Full Model

In [ ]:
# Save the full model (including architecture)
model.save(os.path.join(model_dir, 'bert_full_model.h5'))
print("Full model saved to results/models/bert_full_model.h5")

## 14. Model Predictions on Test Data

In [ ]:
# Show predictions on actual test data
print("\n" + "=" * 80)
print("SAMPLE PREDICTIONS FROM TEST SET")
print("=" * 80)

# Display first 10 test samples with predictions
num_samples = min(10, len(X_test))

for idx in range(num_samples):
    text = X_test[idx]
    true_label = "Fake" if y_test[idx] == 1 else "Not Fake"
    pred_label = "Fake" if y_pred[idx] == 1 else "Not Fake"
    confidence = np.max(y_pred_probs[idx]) * 100
    match = "✓" if y_pred[idx] == y_test[idx] else "✗"
    
    print(f"\n[Sample {idx+1}] {match}")
    print(f"Text: {text[:80]}...")
    print(f"True Label:      {true_label}")
    print(f"Predicted Label: {pred_label} ({confidence:.2f}% confidence)")

print("\n" + "=" * 80)
print(f"\nTotal Test Samples: {len(X_test)}")
print(f"Correct Predictions: {(y_pred == y_test).sum()} / {len(y_test)}")